In [20]:
import os
from pyspark.sql import functions as F

BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

# Load the clean stations we just saved
df_stations = spark.read.parquet(
    os.path.join(OUTPUT_DIR, "cleaned_stations.parquet")
)

# Load raw census data
df_census_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(os.path.join(BASE_DIR, "acs_zcta_combined.csv"))

print("Stations:", df_stations.count())
print("Census ZCTAs:", df_census_raw.count())

Stations: 78085
Census ZCTAs: 33772


In [21]:
# The census zcta is an integer like 601
# We need it as a zero-padded string like "00601"
# to match the AFDC ZIP format

df_census_clean = df_census_raw \
    .withColumn(
        "zcta_str",
        F.lpad(F.col("zcta").cast("string"), 5, "0")
    ) \
    .drop("zcta") \
    .withColumnRenamed("zcta_str", "zcta")

# Verify the fix
print("=== Sample Census ZCTAs after padding ===")
df_census_clean.show(5)

# Check for nulls in census key columns
print("\nCensus null check:")
for col_name in ["zcta", "total_population", "median_household_income"]:
    n = df_census_clean.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name}: {n} nulls")

=== Sample Census ZCTAs after padding ===
+----------------+-----------------------+-----+
|total_population|median_household_income| zcta|
+----------------+-----------------------+-----+
|           16721|                18571.0|00601|
|           37510|                21702.0|00602|
|           48317|                19243.0|00603|
|            5435|                20226.0|00606|
|           25413|                23732.0|00610|
+----------------+-----------------------+-----+
only showing top 5 rows

Census null check:
  zcta: 0 nulls
  total_population: 0 nulls
  median_household_income: 3154 nulls


In [22]:
# Left join - keep ALL stations even if they don't match a Census ZCTA
# We use left join (not inner join) because:
# - An inner join would silently drop stations with no Census match
# - We want to know HOW MANY stations couldn't be matched
# - Geographic analysis still works without income/population data

df_joined = df_stations.join(
    df_census_clean,
    df_stations["ZIP"] == df_census_clean["zcta"],
    how="left"
).drop("zcta")  # drop duplicate key column after join

print(f"Joined row count: {df_joined.count()}")
print(f"Original station count: {df_stations.count()}")
print(f"Difference (should be 0 - left join keeps all): "
      f"{df_stations.count() - df_joined.count()}")

[Stage 117:>                                                        (0 + 1) / 1]

Joined row count: 78085
Original station count: 78085
Difference (should be 0 - left join keeps all): 0


In [23]:
# How many stations successfully matched to Census data?
# How many didn't?

total = df_joined.count()
matched = df_joined.filter(F.col("median_household_income").isNotNull()).count()
unmatched = df_joined.filter(F.col("median_household_income").isNull()).count()

print(f"Total stations:     {total:,}")
print(f"Matched to Census:  {matched:,} ({matched/total*100:.1f}%)")
print(f"Unmatched:          {unmatched:,} ({unmatched/total*100:.1f}%)")

# Where are the unmatched stations located?
print("\n=== Unmatched stations by state (top 10) ===")
df_joined.filter(F.col("median_household_income").isNull()) \
    .groupBy("State").count() \
    .orderBy("count", ascending=False) \
    .show(10)

Total stations:     78,085
Matched to Census:  76,695 (98.2%)
Unmatched:          1,390 (1.8%)

=== Unmatched stations by state (top 10) ===
+-----+-----+
|State|count|
+-----+-----+
|   CA|  392|
|   NY|  111|
|   FL|   72|
|   GA|   64|
|   MA|   60|
|   WA|   55|
|   CT|   49|
|   TX|   46|
|   AZ|   43|
|   IL|   33|
+-----+-----+
only showing top 10 rows


In [24]:
JOINED_PATH = os.path.join(OUTPUT_DIR, "stations_with_census.parquet")

df_joined.write \
    .mode("overwrite") \
    .parquet(JOINED_PATH)

print(f"Saved to: {JOINED_PATH}")

verify = spark.read.parquet(JOINED_PATH)
print(f"Verification row count: {verify.count()}")
print(f"Columns: {verify.columns}")

Saved to: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer/output/stations_with_census.parquet
Verification row count: 78085
Columns: ['ID', 'Station Name', 'City', 'State', 'ZIP', 'Latitude', 'Longitude', 'Access Code', 'Status Code', 'Open Date', 'EV Level1 EVSE Num', 'EV Level2 EVSE Num', 'EV DC Fast Count', 'EV Network', 'EV Connector Types', 'Country', 'open_date_parsed', 'install_year', 'install_month', 'total_ports', 'charger_level', 'is_dcfc', 'region', 'total_population', 'median_household_income']
